# PGM Forcing Generator (Native Tool)

This notebook implements a **native forcing generator** for non-climate forcings without using MIKE ToolboxShell.

What it does:
- Read **one YAML config that defines any number of forcings**, each with:
  - forcing name
  - input grid-code DFS2 path
  - per-grid-code input timeseries (DFS0 or CSV)
  - output DFS2 path
  - output item metadata (name, EUMType, EUMUnit)
- Read grid codes from DFS2
- Read and standardize all input timeseries
- Apply timeseries values to all cells sharing each grid code
- Write one time-varying DFS2 output per forcing directly with `mikeio`

Rules:
- Settings shared by all forcings go under `defaults:` in the YAML; a forcing overrides only what differs
- Grid codes present in the grid DFS2 but **not** listed for a forcing are filled with a constant value of **0** (no error)
- At least one grid code per forcing must have a timeseries
- Missing timesteps in each series are backfilled after alignment
- Near-daily timestamps with minor hour noise are normalized to daily to avoid excessive timesteps

Notes:
- Climate forcings remain out of scope here (provided through PDP repository).
- Paths may be absolute or relative to the module root.

## Step 1: Environment and Module Import

This step configures notebook-relative paths and imports the native forcing-generator module from src.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import mikeio
from plant_growth_module import forcing_generator_native  # noqa: E402

REPO_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
SRC_DIR = REPO_ROOT.joinpath("src")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


MODULE_ROOT = REPO_ROOT
print("Repo root:   ", REPO_ROOT)

## Step 2: Configure Forcings

Point the notebook at the YAML config that defines the forcings. Everything a forcing needs — grid-code DFS2, output DFS2, item name, EUM type/unit and its per-grid-code timeseries — is edited in that file, so adding a forcing means adding a YAML entry, not editing this notebook.

Config layout (see `sample_data/pgm_forcing_generator/timeseries_inputs.yaml`):

```yaml
defaults:
  grid_code_dfs2: sample_data/.../GridCode5.dfs2
  output_dir: output_data/pgm_forcing_generator/forcing_generator

timeseries_inputs:
  var1:
    output_grid: var1.dfs2
    item_name: Seed Application Rate
    eum_type: Concentration
    eum_unit: kg_per_meter_pow_3
    inputs:
      - grid_code: 1
        path: ...
```

Notes:
- `output_grid` defaults to `<forcing name>.dfs2` and is written into `output_dir` when relative.
- `item_name` defaults to the forcing name.
- Grid codes found in the grid-code DFS2 but omitted from a forcing are filled with 0.

Tip: If you are unsure which EUM type or EUM unit values are valid, use the utility section further down in this notebook (Utilities: EUM Type and Unit Lookup).

In [ ]:
# All forcings are defined in a single YAML config: one named entry per forcing,
# each with its own output DFS2, item metadata and per-grid-code timeseries.
# Shared settings (grid-code DFS2, output folder, EUM defaults) live under
# `defaults:` in the same file. Grid codes present in the grid-code DFS2 but not
# listed for a forcing are automatically filled with a constant value of 0.
FORCING_CONFIG = MODULE_ROOT.joinpath(
    "sample_data",
    "pgm_forcing_generator",
    "timeseries_inputs.yaml",
)

FORCINGS = forcing_generator_native.load_forcing_configs(MODULE_ROOT, FORCING_CONFIG)

# Stop at the first failing forcing, or run the rest and report failures at the end.
CONTINUE_ON_ERROR = False

print("Forcing config (YAML):   ", FORCING_CONFIG)
print("Number of forcings:      ", len(FORCINGS))
for forcing in FORCINGS:
    print(
        f"  - {forcing['name']}: {len(forcing['timeseries_inputs'])} timeseries input(s)"
        f" -> {forcing['output_grid']}"
    )

## Step 3: Validate Configuration

Resolve each forcing (its YAML entry merged with `defaults:`), check required files/folders, and print the concrete run inputs per forcing.

In [ ]:
forcing_rows = forcing_generator_native.build_forcing_rows(MODULE_ROOT, FORCINGS)

for row in forcing_rows:
    print(f"Forcing: {row['name']}")
    if row["error"]:
        print(f"  CONFIG ERROR: {row['error']}")
        print()
        continue

    print(f"  Grid-code DFS2:      {row['grid_code_dfs2']}")
    print(f"  Grid-code exists:    {row['grid_code_dfs2_exists']}")
    print(f"  Output grid:         {row['output_grid']}")
    print(f"  Output item:         {row['item_name']}")
    print(f"  Output EUM:          {row['eum_type']} / {row['eum_unit']}")
    print("  Timeseries inputs:")
    for ts_row in row["timeseries_inputs"]:
        print(
            f"    grid_code={ts_row['grid_code']}, source={ts_row['source']}, "
            f"item={ts_row['item']}, exists={ts_row['path_exists']}"
        )
        print(f"      path: {ts_row['path']}")
    print()

## Step 4: Run Native Forcing Generation

Execute the native generator for every configured forcing and write one output DFS2 grid series each.

In [ ]:
config_errors = [row for row in forcing_rows if row["error"]]
missing_grids = [
    row for row in forcing_rows if not row["error"] and not row["grid_code_dfs2_exists"]
]
missing_inputs = [
    row for row in forcing_rows if not row["error"] and row["missing_timeseries_inputs"]
]

if config_errors:
    details = "\n".join(f"{row['name']}: {row['error']}" for row in config_errors)
    raise ValueError("Incomplete forcing configuration:\n" + details)

if missing_grids:
    details = "\n".join(f"{row['name']}: {row['grid_code_dfs2']}" for row in missing_grids)
    raise FileNotFoundError("Missing grid-code DFS2 file(s):\n" + details)

if missing_inputs:
    details = "\n".join(
        f"{row['name']}: {path}"
        for row in missing_inputs
        for path in row["missing_timeseries_inputs"]
    )
    raise FileNotFoundError("Missing timeseries input files:\n" + details)

run_results = forcing_generator_native.run_forcing_setups(
    module_root=MODULE_ROOT,
    forcings=FORCINGS,
    continue_on_error=CONTINUE_ON_ERROR,
)

n_ok = sum(1 for result in run_results if result["status"] == "ok")
print(f"Native forcing generation completed: {n_ok}/{len(run_results)} forcing(s) written.")
for result in run_results:
    if result["status"] != "ok":
        print(f"  {result['name']}: FAILED - {result['error']}")
        continue
    print(f"  {result['name']}: {result['output_grid']}")
    print(
        f"    timesteps={result['n_timesteps']}, "
        f"{result['start_time']} -> {result['end_time']}"
    )

## Step 5: Inspect Run Outputs

Read each generated DFS2 and print key output diagnostics.

### Step 5.A: Output Summary

Run the next cell to inspect written DFS2 metadata and integrity details for every forcing.

In [ ]:
for result in run_results:
    print(f"Forcing: {result['name']}")
    if result["status"] != "ok":
        print(f"  FAILED: {result['error']}")
        print()
        continue

    output_path = Path(result["output_grid"])
    output_exists = output_path.exists()

    if output_exists:
        generated = mikeio.read(output_path)
        generated_da = generated[0]
        written_item_name = generated_da.item.name
        written_timesteps = len(generated_da.time)
    else:
        written_item_name = "<missing>"
        written_timesteps = 0

    print("  Grid code DFS2:        ", result["grid_code_dfs2"])
    print("  Output grid:           ", result["output_grid"])
    print("  Number of grid codes:  ", result["n_grid_codes"])
    print("  Zero-filled codes:     ", result["zero_filled_grid_codes"])
    print("  Timesteps:             ", result["n_timesteps"])
    print("  Start time:            ", result["start_time"])
    print("  End time:              ", result["end_time"])
    print("  Output exists:         ", output_exists)
    print("  Written item name:     ", written_item_name)
    print("  Written timesteps:     ", written_timesteps)
    print()

## Utilities: EUM Type and Unit Lookup

Use this utility to discover valid `eum_type` and `eum_unit` values for the YAML config before running Step 4.

What this helps with:
- Find candidate EUM types by text search
- Find candidate EUM units by text search
- Inspect compatibility between a specific type and unit

How to use:
- Set `EUM_TYPE_QUERY` and/or `EUM_UNIT_QUERY` in the next cell
- Run the next cell to print matching names
- If an exact type/unit is provided, the utility also prints compatible counterparts

### Utility Notes

Suggested workflow:
- Start with a broad query, for example `Concentration` or `kg`
- Copy one exact match into the YAML config (`eum_type` / `eum_unit`)
- Re-run this utility to confirm compatibility when in doubt

Output interpretation:
- `Matched EUM types`: names containing `EUM_TYPE_QUERY`
- `Matched EUM units`: names containing `EUM_UNIT_QUERY`
- `Compatible units`: units valid for an exact type match
- `Compatible types`: types valid for an exact unit match

In [ ]:
# EUM helper: search possible EUM types/units and inspect compatibility.
# Set one or both queries below, then run this cell.
EUM_TYPE_QUERY = ""  # example: "Concentration"
EUM_UNIT_QUERY = ""  # example: "kg_per_meter_pow_3"

matches = forcing_generator_native.eum_matches(
    type_query=EUM_TYPE_QUERY,
    unit_query=EUM_UNIT_QUERY,
)

matched_types = matches["matched_types"]
matched_units = matches["matched_units"]
compatible_units = matches["compatible_units"]
compatible_types = matches["compatible_types"]

print(f"Matched EUM types ({len(matched_types)}):")
print(matched_types[:100])
if len(matched_types) > 100:
    print("... truncated ...")

print()
print(f"Matched EUM units ({len(matched_units)}):")
print(matched_units[:100])
if len(matched_units) > 100:
    print("... truncated ...")

if compatible_units:
    print()
    print(f"Compatible units ({len(compatible_units)}):")
    print(compatible_units)

if compatible_types:
    print()
    print(f"Compatible types ({len(compatible_types)}):")
    print(compatible_types)